# Improvement Experiments - Stage 1 (Few-shot + Error Analysis)

This notebook explores improvements **without changing the main pipeline**. We focus on:
- Few-shot prompting using training queries
- Constraint extraction error analysis
- Iterative prompt tuning (no new models beyond the LLM)


## 0. Setup

Requirements:
```
pip install openai python-dotenv
```


In [1]:
import os
import re
import json
import ast
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

# Load .env (expects OPENROUTER_API_KEY)
load_dotenv(dotenv_path=Path(".env"), override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")

LLM_MODEL = "openai/gpt-5.2"

ARTIFACTS_DIR = "improvement_artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

np.random.seed(42)


In [2]:
train = pd.read_csv("queries_train.csv")
products = pd.read_csv("products.csv")

print(train.head(3))


    query_id                                         query_text  \
0  TRAIN_001  need a pick me up, something like just black c...   
1  TRAIN_002  May I get a chai or tea that's no regular milk...   
2  TRAIN_003  hit me with basic brewed coffee that's under 4...   

  constraint_temperature  constraint_max_calories  constraint_max_sugar  \
0                    NaN                      NaN                  45.0   
1                    NaN                    250.0                   NaN   
2                    NaN                      NaN                  45.0   

  constraint_dairy_free constraint_vegan constraint_caffeine_level  \
0                   NaN              NaN                      high   
1                  True              NaN                       NaN   
2                   NaN              NaN                       NaN   

                                   relevant_products  constraint_max_price  \
0       ['BRW_002', 'BRW_001', 'BRW_005', 'BRW_003']                   

## 1. Constraint Schema + Normalization (same as solution)


In [3]:
VALID_CATEGORIES = {"brewed", "cold_brew", "espresso", "frappuccino", "refresher", "tea"}
VALID_TEMPS = {"hot", "iced", "blended"}
VALID_CAFFEINE = {"none", "low", "medium", "high"}

CATEGORY_ALIASES = {
    "cold brew": "cold_brew",
    "cold-brew": "cold_brew",
    "frap": "frappuccino",
    "frapp": "frappuccino",
    "frappucino": "frappuccino",
    "refresher": "refresher",
    "refreshers": "refresher",
}

TEMP_ALIASES = {
    "cold": "iced",
    "ice": "iced",
    "iced": "iced",
}

CAFFEINE_ALIASES = {
    "decaf": "none",
    "caffeine free": "none",
    "no caffeine": "none",
}

CONSTRAINT_KEYS = [
    "category",
    "temperature",
    "max_calories",
    "max_sugar",
    "max_price",
    "dairy_free",
    "vegan",
    "caffeine_level",
]

SYSTEM_PROMPT = (
    "You are a strict information extraction system. "
    "Extract the constraints with the following keys: "
    "category, temperature, max_calories, max_sugar, max_price, dairy_free, vegan, caffeine_level. "
    "Use null for unknown. "
    "Valid values: category in [brewed, cold_brew, espresso, frappuccino, refresher, tea]; "
    "temperature in [hot, iced, blended]; "
    "caffeine_level in [none, low, medium, high]; "
    "dairy_free and vegan are true/false. "
    "Numbers should be numeric, not strings. "
"Rules: If query mentions black/plain/just coffee (and no espresso terms), set category=brewed. "
"If query says no milk/without milk/no dairy, set dairy_free=true. "
"Only set temperature when the user explicitly asks for hot/iced/blended or mentions ice/blended/frappuccino. "
"Do not infer temperature from weather or feelings (e.g., \"it's hot out\")."
)

def _to_bool(x):
    if isinstance(x, bool):
        return x
    if isinstance(x, str):
        if x.strip().lower() in {"true", "yes", "y"}:
            return True
        if x.strip().lower() in {"false", "no", "n"}:
            return False
    return None

def normalize_constraints(raw: dict) -> dict:
    out = {k: None for k in CONSTRAINT_KEYS}
    for k in CONSTRAINT_KEYS:
        if k in raw:
            out[k] = raw[k]

    for k in CONSTRAINT_KEYS:
        if out[k] is not None and pd.isna(out[k]):
            out[k] = None

    if isinstance(out["category"], str):
        out["category"] = out["category"].strip().lower()
        out["category"] = CATEGORY_ALIASES.get(out["category"], out["category"])
        if out["category"] not in VALID_CATEGORIES:
            out["category"] = None

    if isinstance(out["temperature"], str):
        out["temperature"] = out["temperature"].strip().lower()
        out["temperature"] = TEMP_ALIASES.get(out["temperature"], out["temperature"])
        if out["temperature"] not in VALID_TEMPS:
            out["temperature"] = None

    if isinstance(out["caffeine_level"], str):
        out["caffeine_level"] = out["caffeine_level"].strip().lower()
        out["caffeine_level"] = CAFFEINE_ALIASES.get(out["caffeine_level"], out["caffeine_level"])
        if out["caffeine_level"] not in VALID_CAFFEINE:
            out["caffeine_level"] = None

    out["dairy_free"] = _to_bool(out["dairy_free"])
    out["vegan"] = _to_bool(out["vegan"])

    for k in ["max_calories", "max_sugar", "max_price"]:
        if out[k] is None or out[k] == "":
            out[k] = None
        else:
            try:
                out[k] = float(out[k])
            except Exception:
                out[k] = None

    return out

def constraints_from_train_row(row: pd.Series) -> dict:
    raw = {
        "category": row.get("constraint_category"),
        "temperature": row.get("constraint_temperature"),
        "max_calories": row.get("constraint_max_calories"),
        "max_sugar": row.get("constraint_max_sugar"),
        "max_price": row.get("constraint_max_price"),
        "dairy_free": row.get("constraint_dairy_free"),
        "vegan": row.get("constraint_vegan"),
        "caffeine_level": row.get("constraint_caffeine_level"),
    }
    # heuristic flag for regular/normal caffeine wording
    if isinstance(row.get("query_text"), str):
        q = row.get("query_text").lower()
        if "regular caffeine" in q or "normal caffeine" in q:
            raw["_regular_caffeine"] = True
    return normalize_constraints(raw)


## 2. OpenRouter Client + Structured Output


In [ ]:
from openai import OpenAI


def get_openai_client():
    if not OPENROUTER_API_KEY:
        raise ValueError("OPENROUTER_API_KEY is not set. Add it to .env")
    return OpenAI(api_key=OPENROUTER_API_KEY, base_url=OPENROUTER_BASE_URL)


def extract_constraints_llm(query_text: str, examples: list = None, model: str = LLM_MODEL) -> dict:
    client = get_openai_client()

    schema = {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "category": {
                "type": ["string", "null"],
                "enum": ["brewed", "cold_brew", "espresso", "frappuccino", "refresher", "tea", None],
            },
            "temperature": {
                "type": ["string", "null"],
                "enum": ["hot", "iced", "blended", None],
            },
            "max_calories": {"type": ["number", "null"]},
            "max_sugar": {"type": ["number", "null"]},
            "max_price": {"type": ["number", "null"]},
            "dairy_free": {"type": ["boolean", "null"]},
            "vegan": {"type": ["boolean", "null"]},
            "caffeine_level": {
                "type": ["string", "null"],
                "enum": ["none", "low", "medium", "high", None],
            },
        },
        "required": [
            "category",
            "temperature",
            "max_calories",
            "max_sugar",
            "max_price",
            "dairy_free",
            "vegan",
            "caffeine_level",
        ],
    }

    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]

    if examples:
        ex_lines = ["Examples:"]
        for ex in examples:
            ex_lines.append(f"Query: {ex['query_text']}")
            ex_lines.append(f"Output: {json.dumps(ex['constraints'])}")
        msgs.append({"role": "user", "content": "\n".join(ex_lines)})

    msgs.append({"role": "user", "content": f"Query: {query_text}"})

    response = client.responses.create(
        model=model,
        input=msgs,
        text={
            "format": {
                "type": "json_schema",
                "strict": True,
                "name": "constraint_extraction",
                "schema": schema,
            }
        },
    )
    raw = json.loads(response.output_text)
    if "regular caffeine" in query_text.lower() or "normal caffeine" in query_text.lower():
        raw["_regular_caffeine"] = True
    return normalize_constraints(raw)


## 3. Few-shot Example Selection

We choose examples from training data to use as few-shot demonstrations.


In [5]:
def count_constraints(row: pd.Series) -> int:
    count = 0
    for col in [
        "constraint_temperature",
        "constraint_max_calories",
        "constraint_max_sugar",
        "constraint_dairy_free",
        "constraint_vegan",
        "constraint_caffeine_level",
        "constraint_max_price",
        "constraint_category",
    ]:
        if pd.notna(row.get(col)):
            count += 1
    return count




def select_forced_examples(train_df: pd.DataFrame, forced_ids=None):
    forced_ids = forced_ids or ["TRAIN_031", "TRAIN_085", "TRAIN_050", "TRAIN_020", "TRAIN_024", "TRAIN_070", "TRAIN_018", "TRAIN_009"]
    examples = []
    for qid in forced_ids:
        rows = train_df[train_df["query_id"] == qid]
        if len(rows) == 0:
            continue
        row = rows.iloc[0]
        examples.append({
            "query_text": row["query_text"],
            "constraints": constraints_from_train_row(row),
        })
    return examples


def select_fewshot(train_df: pd.DataFrame, per_category: int = 2) -> list:
    examples = []
    for cat, grp in train_df.groupby("constraint_category"):
        grp = grp.copy()
        grp["constraint_count"] = grp.apply(count_constraints, axis=1)
        grp = grp.sort_values("constraint_count", ascending=False).head(per_category)
        for _, row in grp.iterrows():
            examples.append({
                "query_text": row["query_text"],
                "constraints": constraints_from_train_row(row),
            })
    return examples

fewshot_examples = select_fewshot(train, per_category=2)
forced_examples = select_forced_examples(train)
for ex in forced_examples:
    if ex not in fewshot_examples:
        fewshot_examples.append(ex)
print("fewshot examples:", len(fewshot_examples))


fewshot examples: 20


## 4. Constraint Error Analysis

Run the LLM on a subset of training queries and compare extracted constraints with ground truth.
Results are also saved to `improvement_artifacts/constraint_eval_<n>.jsonl`.


In [6]:
def compare_constraints(pred: dict, truth: dict) -> dict:
    out = {}
    for k in CONSTRAINT_KEYS:
        out[k] = (pred.get(k) == truth.get(k))
    return out


def evaluate_constraints(train_df: pd.DataFrame, n_rows: int = 30, use_fewshot: bool = True, save_artifacts: bool = True) -> pd.DataFrame:
    subset = train_df.sample(n=n_rows, random_state=42).reset_index(drop=True)

    out_dir = Path(ARTIFACTS_DIR)
    out_dir.mkdir(exist_ok=True)
    out_path = out_dir / f"constraint_eval_{n_rows}.jsonl"
    rows = []
    for _, row in subset.iterrows():
        truth = constraints_from_train_row(row)
        pred = extract_constraints_llm(
            row["query_text"],
            examples=fewshot_examples if use_fewshot else None,
        )
        comp = compare_constraints(pred, truth)
        row_obj = {
            "query_id": row["query_id"],
            "query_text": row["query_text"],
            "pred": pred,
            "truth": truth,
            **{f"ok_{k}": v for k, v in comp.items()},
        }
        if save_artifacts:
            with out_path.open("a") as f:
                f.write(json.dumps(row_obj) + "\n")
        rows.append(row_obj)
    return pd.DataFrame(rows)


def summarize_accuracy(df: pd.DataFrame):
    cols = [c for c in df.columns if c.startswith("ok_")]
    return df[cols].mean().sort_values(ascending=False)


In [7]:
# Run evaluation on train set
results = evaluate_constraints(train, n_rows=100, use_fewshot=True, save_artifacts=True)
summarize_accuracy(results)

ok_category          1.0
ok_temperature       1.0
ok_max_calories      1.0
ok_max_sugar         1.0
ok_max_price         1.0
ok_dairy_free        1.0
ok_vegan             1.0
ok_caffeine_level    1.0
dtype: float64

## 5. Inspect Errors (Optional)

Use this to find patterns in the mistakes.


In [ ]:
# Inspect some bad cases
# results = evaluate_constraints(train, n_rows=100, use_fewshot=True)
bad = results[results[[c for c in results.columns if c.startswith('ok_')]].sum(axis=1) < len(CONSTRAINT_KEYS)]
bad[["query_id", "query_text", "pred", "truth"]].head(10)

,query_id,query_text,pred,truth


In [34]:
# Load the most recent results file
import json, pandas as pd
from pathlib import Path

path = Path("improvement_artifacts/constraint_eval_100.jsonl")
rows = [json.loads(l) for l in path.read_text().splitlines() if l.strip()]
df = pd.DataFrame(rows)

# Show any category mistakes
bad_cat = df[df["ok_category"] == False]
bad_cat[["query_id", "query_text", "pred", "truth"]].head(10)


,query_id,query_text,pred,truth


## 6. Stage 2 + Stage 3 Experiment (Enriched Product Text)

This section evaluates whether enriching product text improves ranking, while keeping **embeddings + similarity only**.


In [6]:
# Load stage 1 results from solution artifacts

def load_constraints_from_solution_artifacts(query_id: str) -> dict:
    """
    Load pre-computed constraints from solution/artifacts/{query_id}/stage1_constraints.json
    """
    artifact_path = os.path.join("artifacts", f"{query_id}", "stage1_constraints.json")
    if not os.path.exists(artifact_path):
        return None
    with open(artifact_path) as f:
        return json.load(f)

In [7]:
# Stage 2: filter products using constraints (same logic as solution)

SOFT_CAFFEINE_FILTER = False
CAFFEINE_LOW_MAX = 60 #95
CAFFEINE_MED_MAX = 140 #200


def caffeine_bucket(mg: float) -> str:
    if mg == 0:
        return "none"
    if mg <= CAFFEINE_LOW_MAX:
        return "low"
    if mg <= CAFFEINE_MED_MAX:
        return "medium"
    return "high"


def apply_constraints(df: pd.DataFrame, c: dict) -> pd.DataFrame:
    out = df.copy()

    if c.get("category"):
        out = out[out["category"] == c["category"]]
    if c.get("temperature"):
        out = out[out["temperature"] == c["temperature"]]
    if c.get("max_calories") is not None:
        out = out[out["calories"] <= c["max_calories"]]
    if c.get("max_sugar") is not None:
        out = out[out["sugar_g"] <= c["max_sugar"]]
    if c.get("max_price") is not None:
        out = out[out["price"] <= c["max_price"]]

    if c.get("dairy_free") is True:
        out = out[out["contains_dairy"] == False]
    if c.get("vegan") is True:
        out = out[out["is_vegan"] == True]

    if c.get("caffeine_level") and not SOFT_CAFFEINE_FILTER:
        # If query implies regular/normal caffeine, skip hard filtering
        if c.get("_regular_caffeine") is not True:
            level = c["caffeine_level"]
            out = out[out["caffeine_mg"].apply(caffeine_bucket) == level]

    return out


# Debug helper: show how each constraint reduces candidates
def filter_trace(df: pd.DataFrame, c: dict):
    out = df.copy()
    steps = []

    def log(label, before, after):
        steps.append({"step": label, "before": len(before), "after": len(after)})

    if c.get("category"):
        before = out
        out = out[out["category"] == c["category"]]
        log(f"category == {c['category']}", before, out)

    if c.get("temperature"):
        before = out
        out = out[out["temperature"] == c["temperature"]]
        log(f"temperature == {c['temperature']}", before, out)

    if c.get("max_calories") is not None:
        before = out
        out = out[out["calories"] <= c["max_calories"]]
        log(f"calories <= {c['max_calories']}", before, out)

    if c.get("max_sugar") is not None:
        before = out
        out = out[out["sugar_g"] <= c["max_sugar"]]
        log(f"sugar_g <= {c['max_sugar']}", before, out)

    if c.get("max_price") is not None:
        before = out
        out = out[out["price"] <= c["max_price"]]
        log(f"price <= {c['max_price']}", before, out)

    if c.get("dairy_free") is True:
        before = out
        out = out[out["contains_dairy"] == False]
        log("contains_dairy == False", before, out)

    if c.get("vegan") is True:
        before = out
        out = out[out["is_vegan"] == True]
        log("is_vegan == True", before, out)

    if c.get("caffeine_level") and not SOFT_CAFFEINE_FILTER:
        before = out
        level = c["caffeine_level"]
        out = out[out["caffeine_mg"].apply(caffeine_bucket) == level]
        log(f"caffeine_level == {level}", before, out)

    return out, pd.DataFrame(steps)


def debug_filter_for_query(query_id: str):
    row = train[train['query_id'] == query_id].iloc[0]
    # c = constraints_from_train_row(row)
    c = load_constraints_from_solution_artifacts(query_id)
    filtered, trace = filter_trace(products, c)
    print(row['query_text'])
    print(c)
    display(trace)
    return filtered


In [8]:
filtered = debug_filter_for_query("TRAIN_002")
filtered

May I get a chai or tea that's no regular milk and under 250 cal?
{'category': 'tea', 'temperature': None, 'max_calories': 250.0, 'max_sugar': None, 'max_price': None, 'dairy_free': True, 'vegan': None, 'caffeine_level': None}


,step,before,after
0,category == tea,115,22
1,calories <= 250.0,22,22
2,contains_dairy == False,22,15


,product_id,name,category,subcategory,temperature,caffeine_mg,calories,sugar_g,protein_g,contains_dairy,contains_nuts,contains_gluten,is_vegan,description,price
61,TEA_004,Earl Grey Tea,tea,brewed,hot,40,0,0,0,False,False,False,True,Bold black tea with citrus bergamot,2.95
62,TEA_005,English Breakfast Tea,tea,brewed,hot,40,0,0,0,False,False,False,True,"Robust, full-bodied black tea",2.95
63,TEA_006,Emperors Clouds and Mist Green Tea,tea,brewed,hot,16,0,0,0,False,False,False,True,Light green tea with vegetal notes,2.95
64,TEA_007,Mint Majesty Herbal Tea,tea,herbal,hot,0,0,0,0,False,False,False,True,Refreshing blend of mint and lemon verbena,2.95
65,TEA_008,Peach Tranquility Herbal Tea,tea,herbal,hot,0,0,0,0,False,False,False,True,"Herbal tea with peach, candied pineapple, and ...",2.95
66,TEA_009,Honey Citrus Mint Tea,tea,specialty,hot,16,130,30,1,False,False,False,True,Jade Citrus Mint and Peach Tranquility teas wi...,4.25
67,TEA_010,Matcha Green Tea Latte with Oatmilk,tea,latte,hot,80,210,25,4,False,False,False,True,Matcha sweetened and blended with steamed oat ...,6.25
68,TEA_011,Chai Tea Latte with Oatmilk,tea,latte,hot,95,220,41,3,False,False,False,False,Chai spices with steamed oat milk,5.75
73,ICT_004,Iced Passion Tango Tea,tea,brewed,iced,0,0,0,0,False,False,False,True,"Herbal tea with hibiscus, lemongrass, and apple",3.25
74,ICT_005,Iced Black Tea,tea,brewed,iced,25,0,0,0,False,False,False,True,Premium black tea served over ice,3.25


In [ ]:
# Stage 3: embeddings + similarity only

EMBEDDING_MODEL = "openai/text-embedding-3-large"
USE_ENRICHED_TEXT = True
USE_SUBCATEGORY_TAGS = False
FULL_SIM_WEIGHT = 0.0
TAGS_SIM_WEIGHT = 1.0


def bucketize(value, bins, labels):
    for b, label in zip(bins, labels):
        if value <= b:
            return label
    return labels[-1]


def build_product_text_basic(row: pd.Series) -> str:
    dairy = "dairy" if row["contains_dairy"] else "no dairy"
    vegan = "vegan" if row["is_vegan"] else "not vegan"
    return (
        f"{row['name']}. {row['description']}. "
        f"Category: {row['category']}; Subcategory: {row['subcategory']}; "
        f"Temperature: {row['temperature']}; Caffeine: {row['caffeine_mg']} mg; "
        f"Calories: {row['calories']}; Sugar: {row['sugar_g']} g; Protein: {row['protein_g']} g; "
        f"{dairy}; {vegan}; Price: ${row['price']}"
    )



def extract_product_tags(row: pd.Series) -> list:
    text = f"{row['name']} {row['description']}".lower()
    tags = []
    tags.append(f"subcategory_{row['subcategory'].lower()}")
    if "green" in text and "tea" in text:
        tags.append("green_tea")
    if "herbal" in text:
        tags.append("herbal_tea")
    if "chai" in text:
        tags.append("chai")
    if "matcha" in text:
        tags.append("matcha")
    if "black tea" in text:
        tags.append("black_tea")
    if "fruit" in text or "fruity" in text:
        tags.append("fruity")
    if "refresh" in text:
        tags.append("refreshing")
    return tags


def extract_query_tags(query_text: str) -> list:
    q = query_text.lower()
    tags = []
    if "green tea" in q:
        tags.append("green_tea")
    if "herbal" in q and "tea" in q:
        tags.append("herbal_tea")
    if "chai" in q:
        tags.append("chai")
    if "matcha" in q:
        tags.append("matcha")
    if "black coffee" in q or "just coffee" in q or "plain coffee" in q:
        tags.append("black_coffee")

    # subcategory keyword tags (only if they match actual subcategory labels)
    if USE_SUBCATEGORY_TAGS:
        if "latte" in q:
            tags.append("subcategory_latte")
        if "cappuccino" in q:
            tags.append("subcategory_cappuccino")
        if "americano" in q:
            tags.append("subcategory_americano")
        if "mocha" in q:
            tags.append("subcategory_mocha")
        if "macchiato" in q:
            tags.append("subcategory_macchiato")
        if "flat white" in q:
            tags.append("subcategory_flat_white")
        if "cortado" in q:
            tags.append("subcategory_cortado")
        if "nitro" in q:
            tags.append("subcategory_nitro")
        if "lemonade" in q:
            tags.append("subcategory_lemonade")
        if "iced coffee" in q:
            tags.append("subcategory_iced_coffee")
    if "not too much caffeine" in q or "mild" in q:
        tags.append("caffeine_low")
    if "steaming" in q or "hot" in q:
        tags.append("temperature_hot")
    return tags



def build_product_text_enriched(row: pd.Series) -> str:
    dairy = "dairy_free" if not row["contains_dairy"] else "contains_dairy"
    vegan = "vegan" if row["is_vegan"] else "not_vegan"
    caffeine = caffeine_bucket(row["caffeine_mg"])

    cal_bucket = bucketize(row["calories"], [100, 200, 350], ["low_cal", "mid_cal", "high_cal", "very_high_cal"])
    sugar_bucket = bucketize(row["sugar_g"], [10, 25, 45], ["low_sugar", "mid_sugar", "high_sugar", "very_high_sugar"])
    price_bucket = bucketize(row["price"], [3.5, 5.0, 6.5], ["low_price", "mid_price", "high_price", "very_high_price"])

    tags = [dairy, vegan, f"caffeine_{caffeine}", cal_bucket, sugar_bucket, price_bucket]
    tags += extract_product_tags(row)

    return (
        f"{row['name']}. {row['description']}. "
        f"Category: {row['category']}; Subcategory: {row['subcategory']}; "
        f"Temperature: {row['temperature']}; Caffeine: {row['caffeine_mg']} mg; "
        f"Calories: {row['calories']}; Sugar: {row['sugar_g']} g; Protein: {row['protein_g']} g; "
        f"Tags: {' '.join(tags)}; Price: ${row['price']}"
    )




# Tags-only representations (for constraint-focused similarity)
def build_product_text_tags(row: pd.Series) -> str:
    dairy = "dairy_free" if not row["contains_dairy"] else "contains_dairy"
    vegan = "vegan" if row["is_vegan"] else "not_vegan"
    caffeine = caffeine_bucket(row["caffeine_mg"])

    cal_bucket = bucketize(row["calories"], [100, 200, 350], ["low_cal", "mid_cal", "high_cal", "very_high_cal"])
    sugar_bucket = bucketize(row["sugar_g"], [10, 25, 45], ["low_sugar", "mid_sugar", "high_sugar", "very_high_sugar"])
    price_bucket = bucketize(row["price"], [3.5, 5.0, 6.5], ["low_price", "mid_price", "high_price", "very_high_price"])

    tags = [
        f"category_{row['category']}",
        f"temperature_{row['temperature']}",
        dairy, vegan,
        f"caffeine_{caffeine}",
        cal_bucket, sugar_bucket, price_bucket,
    ]
    tags += extract_product_tags(row)

    # repeat key tags to increase weight
    tags += [dairy, vegan, f"caffeine_{caffeine}"]

    return " ".join(tags)


def build_query_text_tags(query_text: str, c: dict) -> str:
    tags = []
    if c.get("category"):
        tags.append(f"category_{c['category']}")
    if c.get("temperature"):
        tags.append(f"temperature_{c['temperature']}")
    if c.get("dairy_free") is True:
        tags.append("dairy_free")
    if c.get("vegan") is True:
        tags.append("vegan")
    if c.get("caffeine_level"):
        tags.append(f"caffeine_{c['caffeine_level']}")
    if c.get("max_calories") is not None:
        tags.append("low_cal")
    if c.get("max_sugar") is not None:
        tags.append("low_sugar")
    if c.get("max_price") is not None:
        tags.append("low_price")

    # add tag expansions from query text
    tags += extract_query_tags(query_text)

    # repeat key tags to increase weight
    if c.get("caffeine_level"):
        tags.append(f"caffeine_{c['caffeine_level']}")
    if c.get("dairy_free") is True:
        tags.append("dairy_free")

    return " ".join(tags)



def build_query_text(query_text: str, c: dict) -> str:
    parts = [query_text]
    parts += extract_query_tags(query_text)
    if c.get("category"):
        parts.append(f"category {c['category']}")
    if c.get("temperature"):
        parts.append(f"temperature {c['temperature']}")
    if c.get("max_calories") is not None:
        parts.append(f"max calories {c['max_calories']}")
    if c.get("max_sugar") is not None:
        parts.append(f"max sugar {c['max_sugar']}")
    if c.get("max_price") is not None:
        parts.append(f"max price {c['max_price']}")
    if c.get("dairy_free") is True:
        parts.append("dairy free")
    if c.get("vegan") is True:
        parts.append("vegan")
    if c.get("caffeine_level"):
        parts.append(f"caffeine {c['caffeine_level']}")
    return " | ".join(parts)


def l2_normalize(x: np.ndarray) -> np.ndarray:
    if x.ndim == 1:
        x = x.reshape(1, -1)
    norms = np.linalg.norm(x, axis=1, keepdims=True) + 1e-12
    return x / norms


def cosine_similarity(query_vec: np.ndarray, doc_vecs: np.ndarray) -> np.ndarray:
    q = l2_normalize(query_vec)[0]
    d = l2_normalize(doc_vecs)
    return d @ q


class OpenAIEmbedder:
    def __init__(self, model: str = EMBEDDING_MODEL, batch_size: int = 64):
        self.client = get_openai_client()
        self.model = model
        self.batch_size = batch_size

    def embed_texts(self, texts):
        vectors = []
        for i in range(0, len(texts), self.batch_size):
            batch = texts[i : i + self.batch_size]
            resp = self.client.embeddings.create(
                model=self.model,
                input=batch,
            )
            vectors.extend([d.embedding for d in resp.data])
        return np.array(vectors, dtype=np.float32)


def prepare_product_embeddings(products_df: pd.DataFrame):
    if USE_ENRICHED_TEXT:
        texts_full = products_df.apply(build_product_text_enriched, axis=1).tolist()
        suffix = "enriched"
    else:
        texts_full = products_df.apply(build_product_text_basic, axis=1).tolist()
        suffix = "basic"

    texts_tags = products_df.apply(build_product_text_tags, axis=1).tolist()

    safe_model = EMBEDDING_MODEL.replace("/", "_")
    caff_key = f"caff{int(CAFFEINE_LOW_MAX)}_{int(CAFFEINE_MED_MAX)}"
    cache_path_full = Path("cache") / f"product_embeddings_{safe_model}_{suffix}_{caff_key}.npz"
    cache_path_tags = Path("cache") / f"product_embeddings_{safe_model}_{suffix}_{caff_key}_tags.npz"

    if cache_path_full.exists():
        cache = np.load(cache_path_full, allow_pickle=True)
        embeds_full = cache["embeddings"]
    else:
        embedder = OpenAIEmbedder(model=EMBEDDING_MODEL)
        embeds_full = embedder.embed_texts(texts_full)
        cache_path_full.parent.mkdir(exist_ok=True)
        np.savez(cache_path_full, embeddings=embeds_full)
    
    if cache_path_tags.exists():
        cache = np.load(cache_path_tags, allow_pickle=True)
        embeds_tags = cache["embeddings"]
    else:
        embedder = OpenAIEmbedder(model=EMBEDDING_MODEL)
        embeds_tags = embedder.embed_texts(texts_tags)
        cache_path_tags.parent.mkdir(exist_ok=True)
        np.savez(cache_path_tags, embeddings=embeds_tags)

    return embeds_full, embeds_tags

In [ ]:
# Ranking + evaluation on training data using ground-truth constraints

def ndcg_at_k(pred_ids, true_ids, k=None):
    # Graded relevance based on ground-truth order (higher rank = higher relevance)
    if k is None:
        k = len(pred_ids)
    pred_ids = pred_ids[:k]

    rel_map = {pid: len(true_ids) - i for i, pid in enumerate(true_ids)}

    def dcg(ids):
        score = 0.0
        for i, pid in enumerate(ids):
            rel = rel_map.get(pid, 0)
            if rel == 0:
                continue
            score += rel / np.log2(i + 2)
        return score

    ideal = dcg(true_ids[:k])
    if ideal == 0:
        return 0.0
    return dcg(pred_ids) / ideal


def rank_candidates(query_text: str, constraints: dict, candidates: pd.DataFrame, product_embeds_full, product_embeds_tags, id_to_idx) -> list:
    q_text_full = build_query_text(query_text, constraints)
    q_text_tags = build_query_text_tags(query_text, constraints)

    embedder = OpenAIEmbedder(model=EMBEDDING_MODEL)
    q_vec_full = embedder.embed_texts([q_text_full])
    q_vec_tags = embedder.embed_texts([q_text_tags])

    idxs = [id_to_idx[pid] for pid in candidates["product_id"]]
    cand_full = product_embeds_full[idxs]
    cand_tags = product_embeds_tags[idxs]

    sims_full = cosine_similarity(q_vec_full, cand_full)
    sims_tags = cosine_similarity(q_vec_tags, cand_tags)

    sims = FULL_SIM_WEIGHT * sims_full + TAGS_SIM_WEIGHT * sims_tags

    ranked = candidates.copy()
    ranked["score"] = sims
    ranked = ranked.sort_values("score", ascending=False)
    return ranked["product_id"].tolist()

def evaluate_ranking(train_df: pd.DataFrame, max_rows: int = 50):
    subset = train_df.head(max_rows)

    product_ids = products["product_id"].tolist()
    id_to_idx = {pid: i for i, pid in enumerate(product_ids)}
    product_embeds_full, product_embeds_tags = prepare_product_embeddings(products)

    scores = []
    for _, row in subset.iterrows():
        constraints = constraints_from_train_row(row)
        candidates = apply_constraints(products, constraints)
        if len(candidates) == 0:
            continue
        pred_ids = rank_candidates(row["query_text"], constraints, candidates, product_embeds_full, product_embeds_tags, product_embeds_tags)
        true_ids = ast.literal_eval(row["relevant_products"])
        scores.append(ndcg_at_k(pred_ids, true_ids))

    return float(np.mean(scores)) if scores else 0.0


In [38]:
USE_ENRICHED_TEXT = True
score = evaluate_ranking(train, max_rows=100)
print("NDCG (enriched):", score)

NDCG (enriched): 0.9888176622892346


## 7. Hard-Case Analyzer (Lowest-NDCG Queries)

This helps identify remaining ranking failures and why they happen.
NDCG uses graded relevance from the ground-truth order; items not in ground truth contribute 0.


In [ ]:
from typing import List


def analyze_hard_cases(train_df: pd.DataFrame, max_rows: int = 100):
    subset = train_df.head(max_rows)

    product_ids = products["product_id"].tolist()
    id_to_idx = {pid: i for i, pid in enumerate(product_ids)}
    product_embeds_full, product_embeds_tags = prepare_product_embeddings(products)

    results = []
    for _, row in subset.iterrows():
        # constraints = constraints_from_train_row(row)
        constraints = load_constraints_from_solution_artifacts(row["query_id"])
        candidates = apply_constraints(products, constraints)
        if len(candidates) == 0:
            continue

        pred_ids = rank_candidates(row["query_text"], constraints, candidates, product_embeds_full, product_embeds_tags, id_to_idx)
        true_ids = ast.literal_eval(row["relevant_products"])
        score = ndcg_at_k(pred_ids, true_ids)

        results.append({
            "query_id": row["query_id"],
            "query_text": row["query_text"],
            "ndcg": score,
            "pred_ids": pred_ids[:10],
            "true_ids": true_ids,
        })

    results = sorted(results, key=lambda x: x["ndcg"])
    return results


# Example:
# hard_cases = analyze_hard_cases(train, max_rows=100)


In [88]:
score = evaluate_ranking(train, max_rows=100)

In [89]:
print("NDCG (enriched):", score)

NDCG (enriched): 0.9059025700657986


In [99]:
score2 = evaluate_ranking(train, max_rows=100)
print("NDCG (enriched):", score2)

NDCG (enriched): 0.888818564031537


In [90]:
hard_cases = analyze_hard_cases(train, max_rows=100)

In [91]:
[x for x in hard_cases if x["ndcg"] < 1.0]

[{'query_id': 'TRAIN_066',
  'query_text': "long day... give me a cappuccino or latte that's warm, under 250 cals, and high caffeine",
  'ndcg': np.float64(0.3793973447560664),
  'pred_ids': ['ESP_004',
   'ESP_008',
   'ESP_018',
   'ESP_015',
   'ESP_003',
   'ESP_016',
   'ESP_005',
   'ESP_017',
   'ESP_014',
   'ESP_001'],
  'true_ids': ['ESP_002', 'ESP_001', 'ESP_010', 'ESP_009']},
 {'query_id': 'TRAIN_087',
  'query_text': "hit me with a cappuccino or latte that's max $6.5, oat milk or something dairy free, and lots of caffeine",
  'ndcg': np.float64(0.5537799887788619),
  'pred_ids': ['ICE_018',
   'ICE_017',
   'ESP_013',
   'ESP_018',
   'ESP_017',
   'ESP_001',
   'ESP_002',
   'ICE_001',
   'ESP_014',
   'ICE_016'],
  'true_ids': ['ESP_002',
   'ICE_001',
   'ESP_001',
   'ESP_010',
   'ICE_015',
   'ICE_008',
   'ICE_007',
   'ESP_013',
   'ICE_009']},
 {'query_id': 'TRAIN_006',
  'query_text': "running late but i need a cappuccino or latte that's cold and without milk",
 

In [92]:
len(hard_cases)

99

In [ ]:
# Inspect a single hard case with product attributes

def inspect_case(case: dict):
    print("Query:", case["query_text"])
    print("True IDs:", case["true_ids"])
    print("Pred IDs:", case["pred_ids"])

    display(products[products["product_id"].isin(case["true_ids"])][[
        "product_id", "name", "category", "temperature", "calories", "sugar_g", "contains_dairy", "is_vegan", "price"
    ]])

    display(products[products["product_id"].isin(case["pred_ids"])][[
        "product_id", "name", "category", "temperature", "calories", "sugar_g", "contains_dairy", "is_vegan", "price"
    ]])

# Example:
inspect_case(hard_cases[0])


Query: hey can i get something blended that's blended, regular caffeine, and under 350 cal
True IDs: ['FRP_001', 'FRP_015']
Pred IDs: ['FRP_006']


,product_id,name,category,temperature,calories,sugar_g,contains_dairy,is_vegan,price
42,FRP_001,Coffee Frappuccino,frappuccino,blended,230,46,True,False,5.45
57,FRP_015,Matcha Crème Frappuccino,frappuccino,blended,320,50,True,False,5.75


,product_id,name,category,temperature,calories,sugar_g,contains_dairy,is_vegan,price
48,FRP_006,Espresso Frappuccino,frappuccino,blended,210,40,True,False,5.75


In [45]:
# Grid search caffeine thresholds to maximize NDCG
# WARNING: This will call embeddings for each setting unless cached.

def grid_search_caffeine(thresholds_low, thresholds_med, max_rows=50):
    best = None
    results = []
    for low in thresholds_low:
        for med in thresholds_med:
            if med <= low:
                continue
            # set globals
            global CAFFEINE_LOW_MAX, CAFFEINE_MED_MAX
            CAFFEINE_LOW_MAX = float(low)
            CAFFEINE_MED_MAX = float(med)

            score = evaluate_ranking(train, max_rows=max_rows)
            results.append({
                'low': low,
                'med': med,
                'ndcg': score,
            })
            if best is None or score > best['ndcg']:
                best = results[-1]

    return sorted(results, key=lambda x: x['ndcg'], reverse=True), best

# Example search space (adjust as needed)
lows = [60, 80, 95, 110]
meds = [140, 160, 180, 200]
results, best = grid_search_caffeine(lows, meds, max_rows=50)
best


{'low': 60, 'med': 140, 'ndcg': 1.0}

In [ ]:
# 80/20 split grid search for caffeine thresholds
from sklearn.model_selection import train_test_split

train_split, val_split = train_test_split(train, test_size=0.2, random_state=42)

lows = [60, 80, 95, 110]
meds = [140, 160, 180, 200]

results, best = grid_search_caffeine(lows, meds, max_rows=len(train_split))
print('Best on train split:', best)

# Evaluate best on validation split
CAFFEINE_LOW_MAX = best['low']
CAFFEINE_MED_MAX = best['med']
val_score = evaluate_ranking(val_split, max_rows=len(val_split))
print('Validation NDCG:', val_score)

Best on train split: {'low': 60, 'med': 140, 'ndcg': 0.9991906223048999}
Validation NDCG: 0.9977914796615878


In [ ]:
CAFFEINE_LOW_MAX = 60
CAFFEINE_MED_MAX = 140
score = evaluate_ranking(train, max_rows=100)
print(score)

0.9888176622892346


In [50]:
# Compare TAGS_SIM_WEIGHT values on full train set
def compare_tag_weights(weights, max_rows=100):
    results = []
    global FULL_SIM_WEIGHT, TAGS_SIM_WEIGHT
    for w in weights:
        TAGS_SIM_WEIGHT = float(w)
        FULL_SIM_WEIGHT = 1.0 - TAGS_SIM_WEIGHT
        score = evaluate_ranking(train, max_rows=max_rows)
        results.append({
            'tags_weight': TAGS_SIM_WEIGHT,
            'full_weight': FULL_SIM_WEIGHT,
            'ndcg': score,
        })
    return results

# Example:
# weights = [0.2, 0.3, 0.4, 0.5, 0.6]
# compare_tag_weights(weights, max_rows=100)


In [51]:
weights = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
compare_tag_weights(weights, max_rows=100)


[{'tags_weight': 0.0, 'full_weight': 1.0, 'ndcg': 0.9882796374077115},
 {'tags_weight': 0.1, 'full_weight': 0.9, 'ndcg': 0.9882796374077115},
 {'tags_weight': 0.2, 'full_weight': 0.8, 'ndcg': 0.9882796374077115},
 {'tags_weight': 0.3, 'full_weight': 0.7, 'ndcg': 0.9882796374077115},
 {'tags_weight': 0.4, 'full_weight': 0.6, 'ndcg': 0.9882796374077115},
 {'tags_weight': 0.5, 'full_weight': 0.5, 'ndcg': 0.9888176622892346},
 {'tags_weight': 0.6, 'full_weight': 0.4, 'ndcg': 0.9891025305916024},
 {'tags_weight': 0.7,
  'full_weight': 0.30000000000000004,
  'ndcg': 0.9891025305916024},
 {'tags_weight': 0.8,
  'full_weight': 0.19999999999999996,
  'ndcg': 0.9891025305916024},
 {'tags_weight': 0.9,
  'full_weight': 0.09999999999999998,
  'ndcg': 0.9892826139394266},
 {'tags_weight': 1.0, 'full_weight': 0.0, 'ndcg': 0.9893642760546015}]

In [52]:
# Final grid search with current tag/full weights
# Adjust thresholds as needed
lows = [60, 80, 95, 110]
meds = [140, 160, 180, 200]
results, best = grid_search_caffeine(lows, meds, max_rows=100)
best


{'low': 60, 'med': 140, 'ndcg': 0.9893642760546015}

In [ ]:
# Final grid search with current tag/full weights
# Adjust thresholds as needed
lows2 = [10, 20, 30, 40, 50, 60, 70]
meds2 = [120, 130, 140, 160, 180, 200]
results2, best2 = grid_search_caffeine(lows2, meds2, max_rows=100)
best2


In [ ]:
# Weight sweep under graded NDCG
def compare_weights(weights, max_rows=100):
    results = []
    global FULL_SIM_WEIGHT, TAGS_SIM_WEIGHT
    for w in weights:
        TAGS_SIM_WEIGHT = float(w)
        FULL_SIM_WEIGHT = 1.0 - TAGS_SIM_WEIGHT
        score = evaluate_ranking(train, max_rows=max_rows)
        results.append({
            'tags_weight': TAGS_SIM_WEIGHT,
            'full_weight': FULL_SIM_WEIGHT,
            'ndcg': score,
        })
    return results

# Example:
# USE_SUBCATEGORY_TAGS = False
# weights = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
# compare_weights(weights, max_rows=100)
